### Dataset: https://www.kaggle.com/datasets/saisirishan/indian-vehicle-dataset/data

In [1]:
import os

%pwd

'c:\\Users\\Hp\\Documents\\GitHub\\traffic_scene\\notebooks'

In [2]:
os.chdir("../")

%pwd

'c:\\Users\\Hp\\Documents\\GitHub\\traffic_scene'

In [5]:
import random, shutil
from pathlib import Path


# configs
BASE_DIR = Path("dataset/anpr")
OUTPUT_DIR = (BASE_DIR / "cleaned")
FOLDERS = ["google_images", "video_images", "State-wise_OLX"]
SPLIT_RATIOS = (0.8, 0.1, 0.1)  # train, valid, test
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
ANNOTATION_EXT = ".xml"

In [ ]:
def collect_images(base_dir, folders, img_extensions=IMG_EXTENSIONS):
    """Collect all image paths from the specified folders and subfolders."""
    all_images = []
    
    try:
        for folder in folders:
            folder_path = base_dir / folder
            
            if not folder_path.exists():
                print(f"[WARN] Folder not found: {folder_path}")
                continue
            
            for root, _, files in os.walk(folder_path):
                for file in files:
                    if Path(file).suffix.lower() in img_extensions:
                        all_images.append(Path(root) / file)
        
        print(f"[INFO] Total images collected: {len(all_images)}")
    except Exception as e:
        print(f"[ERROR] Failed to collect images: {e}")
    return all_images

def split_dataset(images, ratios):
    """Split image paths into train, valid, and test sets."""
    try:
        random.shuffle(images)
        n_total = len(images)
        
        n_train = int(ratios[0] * n_total)
        n_valid = int(ratios[1] * n_total)
        
        train_set = images[:n_train]
        valid_set = images[n_train:n_train + n_valid]
        test_set = images[n_train + n_valid:]
        
        print(f"[INFO] Split: Train={len(train_set)}, Valid={len(valid_set)}, Test={len(test_set)}")
        return train_set, valid_set, test_set
    except Exception as e:
        print(f"[ERROR] Failed to split dataset: {e}")
        return [], [], []

def copy_files_with_annotations(images, target_dir, annotation_ext=ANNOTATION_EXT):
    """Copy images and their corresponding annotation files to the target directory."""
    try:
        target_dir.mkdir(parents=True, exist_ok=True)
        
        for img_path in images:
            try:
                # copy img
                shutil.copy(img_path, target_dir / img_path.name)
                
                # copy annotation (same name with .xml ext.)
                annotation_path = img_path.with_suffix(annotation_ext)
                
                if annotation_path.exists():
                    shutil.copy(annotation_path, target_dir / annotation_path.name)
                else:
                    print(f"[WARN] No annotation for {img_path.name}")
                
            except Exception as e:
                print(f"[ERROR] Failed to copy {img_path.name}: {e}")
        
    except Exception as e:
        print(f"[ERROR] Failed to copy files to {target_dir}: {e}")


def prepare_dataset(base_dir=BASE_DIR, output_dir=OUTPUT_DIR, folders=FOLDERS, split_ratios=SPLIT_RATIOS):
    """Main pipeline to prepare dataset."""
    try:
        # Step 1: collect all imgs
        images = collect_images(base_dir, folders)
        if not images:
            print("[ERROR] No images found. Exiting...")
            return

        # Step 2: split dataset
        train_set, valid_set, test_set = split_dataset(images, split_ratios)

        # Step 3: copy files
        print("[INFO] Copying Train set...")
        copy_files_with_annotations(train_set, output_dir / "train")

        print("[INFO] Copying Valid set...")
        copy_files_with_annotations(valid_set, output_dir / "valid")

        print("[INFO] Copying Test set...")
        copy_files_with_annotations(test_set, output_dir / "test")

        print("[INFO] Dataset prepared successfully!")
    except Exception as e:
        print(f"[ERROR] Dataset preparation failed: {e}")


prepare_dataset()

[INFO] Total images collected: 1698
[INFO] Split: Train=1358, Valid=169, Test=171
[INFO] Copying Train set...
[INFO] Copying Valid set...
[INFO] Copying Test set...
[INFO] Dataset prepared successfully!


In [6]:
def check_annotations(split_dir, img_extensions=IMG_EXTENSIONS, annotation_ext=ANNOTATION_EXT):
    """Check images and annotation files in the split directory."""
    try:
        print(f"\n[INFO] Checking directory: {split_dir}")

        images = [f for f in split_dir.iterdir() if f.suffix.lower() in img_extensions]
        annotations = [f for f in split_dir.iterdir() if f.suffix.lower() == annotation_ext]

        image_names = {f.stem for f in images}
        annotation_names = {f.stem for f in annotations}

        # find missing annotations
        missing_annotations = image_names - annotation_names
        missing_images = annotation_names - image_names  # shouldn't exist ideally

        print(f"  Total Images: {len(images)}")
        print(f"  Total Annotations: {len(annotations)}")

        if missing_annotations:
            print(f"  [WARN] Missing annotations for {len(missing_annotations)} images:")
            for name in sorted(missing_annotations):
                print(f"    - {name}")
        else:
            print("  [INFO] All images have corresponding annotations.")

        if missing_images:
            print(f"  [WARN] Found annotations with no matching images:")
            for name in sorted(missing_images):
                print(f"    - {name}")

    except Exception as e:
        print(f"[ERROR] Failed to check {split_dir}: {e}")


def verify_dataset(dataset_dir=OUTPUT_DIR):
    """Verify annotations in train, valid, and test directories."""
    try:
        for split in ["train", "valid", "test"]:
            split_dir = dataset_dir / split
            if not split_dir.exists():
                print(f"[WARN] Split directory not found: {split_dir}")
                continue
            check_annotations(split_dir)
        print("\n[INFO] Dataset verification complete.")
    except Exception as e:
        print(f"[ERROR] Dataset verification failed: {e}")


verify_dataset()


[INFO] Checking directory: dataset\anpr\cleaned\train
  Total Images: 1358
  Total Annotations: 1357
  [INFO] All images have corresponding annotations.

[INFO] Checking directory: dataset\anpr\cleaned\valid
  Total Images: 169
  Total Annotations: 169
  [INFO] All images have corresponding annotations.

[INFO] Checking directory: dataset\anpr\cleaned\test
  Total Images: 171
  Total Annotations: 171
  [INFO] All images have corresponding annotations.

[INFO] Dataset verification complete.
